# Import Modules

In [1]:
import sys
from pathlib import Path

# Get the absolute path to the project root
project_root = Path.cwd().parent

# Add the project root to Python's import path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Can now import files as if the notebook was located in the root folder
from src.data.connector_factory import ConnectorFactory
from src.models.bsm import implied_volatility
from src.models.svi import calibrate_raw_svi, svi_iv

import numpy as np
import plotly.graph_objects as go

# SVI Calibration
Check if SVI can be calibrated. The test fetches option data for one expiry using the Connector and fits the volatility surface using SVI

In [4]:
connector = ConnectorFactory.get_connector()
expiries = connector.get_available_expiries()[:3]

# Fetch data
spot = 600.0  # You can fetch this dynamically
r, q = 0.045, 0.015
expiry = expiries[0]
raw = connector.get_chain_for_expiry(expiry)
T = 30/365

print(expiry)

# Compute IVs
strikes, ivs = [], []
for opt in raw["calls"] + raw["puts"]:
    if opt.mid > 0 and opt.bid > 0:
        iv = implied_volatility(opt.mid, spot, opt.strike, T, r, q)
        if not np.isnan(iv):
            strikes.append(opt.strike)
            ivs.append(iv)

# Fit SVI
k = np.log(np.array(strikes) / (spot * np.exp((r - q) * T)))
params = calibrate_raw_svi(k, np.array(ivs), T)
print(f"SVI params: {params}")

2026-07-06
SVI params: [-0.36777227 12.9523434   0.77968229  0.26517783  0.04643901]


In [8]:
# Plot
k_grid = np.linspace(k.min()-0.1, k.max()+0.1, 50)
iv_fitted = svi_iv(k_grid, *params, T)

fig = go.Figure()
fig.add_trace(go.Scatter(x=strikes, y=ivs, mode='markers', name='Market IV'))
fig.add_trace(go.Scatter(
    x=np.exp(k_grid) * (spot * np.exp((r - q) * T)),
    y=iv_fitted,
    mode='lines',
    name='SVI Fit',
))
fig.update_layout(title=f"SVI Fit for {expiry}")
fig.show()